# HealthConnect Rwanda: Week 7 Model Testing and Refinement

**Track:** Data Science  
**Client:** HealthConnect Rwanda  
**Author:** Dorian NJAMI  
**Programme:** AnalystLab Africa Experience Lab  
**Project phase:** Week 7 - Testing, Refinement and End-to-End Validation

## Project Overview

This notebook continues the HealthConnect work from Week 6. It does not repeat the full exploratory analysis, statistical analysis, forecasting, clustering, or business recommendations. It first reviews the Week 6 admission model, checks the data and feature assumptions that may have limited its performance, and builds a more carefully defined candidate. It then evaluates the candidate on a later, time-ordered holdout and documents the remaining validation work.

The assignment describes a Week 5 HealthConnect baseline and a Week 6 improved model. That sequence does not match this project history: HealthConnect work began in Week 6, and the available admission classifier is the Week 6 baseline. It is kept as the historical reference; it is not relabelled as a Week 5 model.

The overall HealthConnect scenario mentions reducing missed appointments. The current integrated dataset has no usable attendance or no-show outcome: its status field contains only Active. The admission model in this notebook therefore remains an admission model and must not be interpreted as a missed-appointment model.

## Notebook Structure

### Part I - Week 6 Conclusions and Adjustments
1. Week 6 model summary and project continuity.
2. Mentor feedback review and evidence-based follow-through.
3. Target and data checks relevant to admission prediction.
4. Review and refinement of modeling features.
5. Development comparison: Week 6-style logistic regression, refined logistic regression, and Random Forest.
6. Train-versus-validation review for possible overfitting and model weakness.

### Part II - Week 7 Testing and Validation
1. Testing readiness and expected outcomes.
2. Retest on a later time period not used for model selection.
3. False-positive, false-negative, and segment analysis.
4. Statistical comparison of refinements, internal model-interface checks, and error-pattern analysis.
5. Cross-track handoff/evidence, model suitability, and the 16-point Week 7 project summary.
6. Limitations, risks, and actions required before Week 8.

Notebook text and code comments are written in English. The model is an analytical prototype and is not a clinical decision tool.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Load the integrated and cleaned Week 6 appointment-level master table.
data_path = Path("../Data/Cleaned_Datasets/healthconnect_master_clean.csv")
if not data_path.exists():
    data_path = Path("Data/Cleaned_Datasets/healthconnect_master_clean.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "The Week 6 cleaned master table was not found. "
        "Run this notebook from the project root or the Notebook folder."
    )

master_df_clean = pd.read_csv(data_path, low_memory=False)
print(f"Loaded {len(master_df_clean):,} appointments from: {data_path}")
print(f"Columns available: {len(master_df_clean.columns)}")

Loaded 5,000 appointments from: ..\Data\Cleaned_Datasets\healthconnect_master_clean.csv
Columns available: 63


## Part I - Week 6 Conclusions and Adjustments

This part establishes a concise baseline, checks the fields that matter for the admission model, and applies evidence-based changes before the Week 7 holdout evaluation. The full Week 6 EDA and unrelated forecasting, clustering, and statistical testing are intentionally omitted.

### 1.1 - Week 6 Model Summary and Continuity

Week 6 produced an appointment-level integrated dataset and a first logistic-regression classifier for admission. The original model retained the first observed appointment per patient, used a random stratified train/test split, and applied class weighting.

On its original test split, the model reported 1,111 observations, an admission prevalence of 11.5%, ROC-AUC of 0.466, balanced accuracy of 0.468, admission recall of 40.6%, and admission precision of 10.1%. It identified 52 admissions and produced 463 false-positive admission alerts. These results do not support operational use.

The Week 7 work will retain that result as the historical Week 6 reference. The controlled model comparison below will use a common temporal development split so that changes to features and algorithms can be compared on the same observations. Because the split differs from Week 6, those controlled results will not be presented as a direct remeasurement of the original Week 6 test score.

**Review of Week 6 testing requirements.** The Week 6 notebook reported a single random holdout evaluation with classification metrics, a classification report, and a confusion matrix. It did not define a separate test plan, time-based validation, train-versus-validation overfitting review, segment analysis, paired uncertainty estimate, or cross-track model test. Those gaps are addressed below where the available data allow. The assignment's Week 5 HealthConnect baseline is not present in this project; this notebook transparently uses the actual Week 6 baseline and a controlled Week 6-style comparator.


### 1.1 - Week 6 Mentor Feedback: Review and Follow-Through

The mentor feedback highlighted four areas: predictive performance and leakage control; facility mapping and financial reconciliation; stronger validation over a longer time series; and reliable features with consistent interpretation across project artifacts. The table below distinguishes actions implemented in this Week 7 notebook from items that remain dependent on data or cross-track evidence. A documented limitation is not treated as a completed fix.

| Mentor feedback | Week 7 action and evidence | Assessment |
|---|---|---|
| Improve predictive performance through feature engineering and model comparison, while checking leakage. | The notebook checks target duplication, removes the copied target and constant field, excludes same-visit tests/prescriptions with unknown timing, adds strictly prior patient history, compares logistic regression and Random Forest against a prior-rate reference, and evaluates on later months. The refined model does not beat the reference on the final test. | **Addressed and tested; improvement not demonstrated.** Do not present the candidate as operationally improved. |
| Resolve department-to-facility mapping and financial reconciliation before operational conclusions. | These are important Week 6 data-integration issues, but this Week 7 notebook tests an admission classifier from the cleaned appointment-level master. It does not rebuild the facility mapping or reconcile billing/revenue source tables. The model conclusions are therefore limited to admission prediction and do not make facility or financial claims. | **Still open for the relevant analytics/financial workstream.** Reconcile source identifiers and totals before using facility or revenue results. |
| Strengthen validation with a longer time series and additional temporal validation. | A chronological train/validation/final-test design is used: January-August, September-October, and November-December 2026. The final period remains untouched until retesting. The available dataset covers only one year, so rolling-origin evaluation across multiple years cannot be performed. | **Partially addressed.** Temporal separation is implemented; longer-history validation remains blocked by data coverage. |
| Improve feature reliability, interpretation, and consistency across notebook, datasets, reports, and presentation. | Feature timing assumptions and excluded fields are documented; preprocessing is fitted within model pipelines; results are interpreted from executed holdout metrics; and the same model scope and limitations are stated throughout this notebook. Diagnosis/severity timestamps, target provenance, and an attendance label are still unverified. | **Partially addressed.** Local definitions are clearer; source-owner and cross-track verification are still required. |

**Overall assessment.** The Week 7 revision responds directly to the modeling and validation concerns, and it finds that the apparent refinement did not produce a reliable gain. The mapping, financial reconciliation, longer time series, and external feature-timing confirmation cannot be claimed as resolved from the files available for this model test. These should remain explicit follow-up items rather than being hidden behind a favorable model narrative.

### 1.2 - Admission Target and Data Readiness Checks

Before changing the model, verify that the target is present and binary, that appointment identifiers are unique, and that dates can support a chronological split. Check class prevalence, repeated appointments per patient, low-variation predictors, and whether an admission source count duplicates the target.

These checks are focused on the modeling component. They do not replace the broader Week 6 source-table quality assessment.

In [2]:
# Check the fields required to define the admission modeling dataset.
required_columns = [
    "appointmentid", "patientid", "date", "admission_indicator",
    "admissions_record_count",
]
missing_columns = [column for column in required_columns
                   if column not in master_df_clean.columns]
if missing_columns:
    raise KeyError(f"Required modeling columns are missing: {missing_columns}")

model_source = master_df_clean.copy()
model_source["date"] = pd.to_datetime(model_source["date"], errors="coerce")
model_source["admission_indicator"] = pd.to_numeric(
    model_source["admission_indicator"], errors="coerce"
)

# Validate target values and the appointment-level grain.
if model_source["date"].isna().any():
    raise ValueError("Some appointment dates could not be parsed.")
if model_source["admission_indicator"].isna().any():
    raise ValueError("The admission target contains missing values.")
if not set(model_source["admission_indicator"].unique()).issubset({0, 1}):
    raise ValueError("The admission target must contain only 0 and 1.")
if not model_source["appointmentid"].is_unique:
    raise ValueError("Appointment IDs are not unique in the master table.")
model_source["admission_indicator"] = (
    model_source["admission_indicator"].astype("int8")
)

# Summarize target prevalence, date coverage, repeated visits, and field variation.
target_summary = pd.DataFrame([{
    "Appointment rows": len(model_source),
    "Distinct patients": model_source["patientid"].nunique(),
    "Unique appointment IDs": model_source["appointmentid"].nunique(),
    "First appointment date": model_source["date"].min().date(),
    "Last appointment date": model_source["date"].max().date(),
    "Admissions": int(model_source["admission_indicator"].sum()),
    "Admission prevalence (%)": 100 * model_source["admission_indicator"].mean(),
    "Rows in repeated patient-date groups": int(
        model_source.duplicated(["patientid", "date"], keep=False).sum()
    ),
}])
display(target_summary.round(2))

# Check whether the admission count is a direct copy of the target.
admission_count_matches_target = model_source[
    "admissions_record_count"
].eq(model_source["admission_indicator"]).all()
print("Admissions record count matches the target on every row:",
      admission_count_matches_target)

# Check attendance information and constant or low-variation predictors.
if "status" in model_source.columns:
    print("Status values:", model_source["status"].value_counts(dropna=False).to_dict())

variation_columns = [
    column for column in [
        "diagnoses_record_count", "laboratory_tests_record_count",
        "prescriptions_record_count", "diagnoses", "diagnosis_severities",
        "visittype", "departmentname",
    ] if column in model_source.columns
]
variation_summary = pd.DataFrame({
    "Distinct values": model_source[variation_columns].nunique(dropna=False),
    "Missing rows": model_source[variation_columns].isna().sum(),
}).sort_values("Distinct values")
display(variation_summary)

,Appointment rows,Distinct patients,Unique appointment IDs,First appointment date,Last appointment date,Admissions,Admission prevalence (%),Rows in repeated patient-date groups
0,5000,4443,5000,2026-01-01,2026-12-28,586,11.72,4


Admissions record count matches the target on every row: True
Status values: {'Active': 5000}


,Distinct values,Missing rows
diagnoses_record_count,1,0
laboratory_tests_record_count,2,0
prescriptions_record_count,2,0
diagnosis_severities,3,0
visittype,3,0
diagnoses,5,0
departmentname,10,0


**Interpretation of the readiness checks.** The integrated table contains 5,000 appointments for 4,443 patients, with dates from January through December 2026. There are 586 admission records, or 11.7% of appointments. The target is therefore imbalanced, so accuracy alone would be misleading.

The audit confirms two important limitations. First, admissions_record_count exactly duplicates admission_indicator and is a direct target leak; it must never be a model input. Second, diagnoses_record_count has only one value across the cohort, so it cannot distinguish admissions from non-admissions and should be removed. The status field is Active for every row, so it cannot define appointment attendance or no-shows.

The original first-appointment-per-patient filter also discarded follow-up appointments. The refined modeling cohort will retain appointment rows and create patient-history variables using information from strictly earlier dates.

### 1.3 - Feature Review and Point-in-Time Refinement

The intended prediction point for this candidate is after the initial clinical assessment and diagnosis, but before tests, prescriptions, or the admission outcome are recorded. This is an explicit modeling assumption because the dataset does not contain event-level timestamps.

The Week 6 specification included test and prescription counts whose availability before the admission decision is not established. The refined specification removes them, removes the constant diagnosis count, and adds prior appointment and prior admission history. History is calculated from dates strictly earlier than the current appointment; appointments on the same date are excluded from one another's history.

Diagnosis and severity remain candidate predictors only under the stated prediction-time assumption. Their availability must be confirmed with the clinical and ML Engineering tracks before any real-world use.

In [3]:
# Create appointment history using only information from earlier calendar dates.
model_source = model_source.sort_values(
    ["patientid", "date", "appointmentid"]
).copy()

patient_day = (
    model_source.groupby(["patientid", "date"], as_index=False)
    .agg(
        appointments_on_date=("appointmentid", "nunique"),
        admissions_on_date=("admission_indicator", "sum"),
    )
    .sort_values(["patientid", "date"])
)
patient_day["previous_appointment_count"] = (
    patient_day.groupby("patientid")["appointments_on_date"].cumsum()
    - patient_day["appointments_on_date"]
)
patient_day["previous_admission_count"] = (
    patient_day.groupby("patientid")["admissions_on_date"].cumsum()
    - patient_day["admissions_on_date"]
)

history_columns = [
    "patientid", "date", "previous_appointment_count",
    "previous_admission_count",
]
model_source = model_source.merge(
    patient_day[history_columns],
    on=["patientid", "date"],
    how="left",
    validate="many_to_one",
)

# A historical admission rate is undefined for patients with no earlier visits.
model_source["previous_admission_rate"] = np.where(
    model_source["previous_appointment_count"] > 0,
    model_source["previous_admission_count"]
    / model_source["previous_appointment_count"],
    np.nan,
)

# Derive simple calendar and ordinal severity features from existing fields.
model_source["appointment_month"] = model_source["date"].dt.month.astype("string")
model_source["appointment_day_of_week"] = (
    model_source["date"].dt.day_name().astype("string")
)
severity_lookup = {"mild": 1, "moderate": 2, "severe": 3}
model_source["disease_severity_score"] = (
    model_source["diagnosis_severities"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .map(severity_lookup)
)

# Confirm that the history features use no current or future appointment events.
if (model_source["previous_appointment_count"] < 0).any():
    raise ValueError("A negative prior appointment count was created.")
if (model_source["previous_admission_count"] < 0).any():
    raise ValueError("A negative prior admission count was created.")
if (model_source["previous_admission_count"]
        > model_source["previous_appointment_count"]).any():
    raise ValueError("Prior admissions exceed prior appointments.")

feature_review = pd.DataFrame([
    {"Feature": "diagnoses_record_count",
     "Week 6 observation": "One distinct value in the full cohort",
     "Week 7 decision": "Remove: no predictive variation"},
    {"Feature": "admissions_record_count",
     "Week 6 observation": "Exactly equals admission_indicator",
     "Week 7 decision": "Exclude: direct target leakage"},
    {"Feature": "laboratory_tests_record_count and prescriptions_record_count",
     "Week 6 observation": "Same-visit events; timing before admission is unknown",
     "Week 7 decision": "Exclude from refined model pending timing evidence"},
    {"Feature": "previous appointment and admission history",
     "Week 6 observation": "First-visit filter made prior visit count uninformative",
     "Week 7 decision": "Recalculate from strictly earlier dates"},
    {"Feature": "diagnosis and severity",
     "Week 6 observation": "Potentially useful, but event timestamps are unavailable",
     "Week 7 decision": "Retain under the stated prediction-time assumption"},
])
display(feature_review)

print(f"Refined modeling table: {len(model_source):,} appointments, "
      f"{model_source['patientid'].nunique():,} patients.")
print("Appointments with at least one earlier appointment:",
      int(model_source["previous_appointment_count"].gt(0).sum()))

,Feature,Week 6 observation,Week 7 decision
0,diagnoses_record_count,One distinct value in the full cohort,Remove: no predictive variation
1,admissions_record_count,Exactly equals admission_indicator,Exclude: direct target leakage
2,laboratory_tests_record_count and prescription...,Same-visit events; timing before admission is ...,Exclude from refined model pending timing evid...
3,previous appointment and admission history,First-visit filter made prior visit count unin...,Recalculate from strictly earlier dates
4,diagnosis and severity,"Potentially useful, but event timestamps are u...",Retain under the stated prediction-time assump...


Refined modeling table: 5,000 appointments, 4,443 patients.
Appointments with at least one earlier appointment: 555


**Interpretation of the feature review.** All 5,000 appointment rows are retained. The revised history fields distinguish first observed encounters from repeat visits and avoid using same-day or future appointments as prior history. The original diagnosis-count feature is dropped because it is constant; the admissions count is prohibited because it reproduces the target; and same-visit tests and prescriptions are left out of the refined model because their timing is not documented.

The revised feature set is more defensible for the defined decision point, but it does not guarantee better predictive performance. The data provide no event timestamps, so diagnosis and severity timing remain assumptions that require cross-track confirmation.

### 1.4 - Development Split and Candidate Model Comparison

Use a chronological development design to better reflect prediction on future appointments. January-August is used for fitting, September-October for model comparison, and November-December is reserved for the Week 7 final retest.

Three model specifications will be compared on the same training and validation periods: the Week 6 predictor specification with logistic regression, a refined logistic regression, and a Random Forest using the refined features. A prior-probability DummyClassifier provides a simple reference. The Week 6-style specification is rerun on the common temporal split and full appointment cohort; it is not the original Week 6 random-split result.

Average precision is the primary selection metric because admissions are the minority class. ROC-AUC, recall, precision, balanced accuracy, and F1 are also reported. The validation period is used for candidate selection; the final test period is not used here.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define time-based partitions. The final two months remain unseen until Part II.
train_data = model_source.loc[model_source["date"] < "2026-09-01"].copy()
validation_data = model_source.loc[
    (model_source["date"] >= "2026-09-01")
    & (model_source["date"] < "2026-11-01")
].copy()
test_data = model_source.loc[model_source["date"] >= "2026-11-01"].copy()

if min(len(train_data), len(validation_data), len(test_data)) == 0:
    raise ValueError("A chronological data partition is empty.")
for name, part in [("Training", train_data), ("Validation", validation_data),
                   ("Final test", test_data)]:
    if part["admission_indicator"].nunique() < 2:
        raise ValueError(f"{name} data do not contain both target classes.")

split_summary = pd.DataFrame([
    {"Partition": name, "Rows": len(part),
     "Start": part["date"].min().date(), "End": part["date"].max().date(),
     "Admission rate (%)": 100 * part["admission_indicator"].mean()}
    for name, part in [("Training", train_data),
                       ("Validation", validation_data),
                       ("Final test", test_data)]
])
display(split_summary.round(2))

# Define the Week 6 predictor specification for a controlled comparison.
week6_numeric_features = [
    "age", "previous_appointment_count", "diagnoses_record_count",
    "laboratory_tests_record_count", "prescriptions_record_count",
    "disease_severity_score",
]
week6_categorical_features = [
    "gender", "visittype", "departmentname", "chroniccondition", "diagnoses",
]

# Define refined predictors available under the stated decision point.
refined_numeric_features = [
    "age", "previous_appointment_count", "previous_admission_count",
    "previous_admission_rate", "disease_severity_score",
]
refined_categorical_features = [
    "gender", "visittype", "departmentname", "chroniccondition",
    "diagnoses", "appointment_month", "appointment_day_of_week",
]

def build_preprocessor(numeric_features, categorical_features, scale_numeric):
    # Create preprocessing steps for numeric and categorical predictors.
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])

def build_logistic_model(numeric_features, categorical_features):
    # Build a class-weighted logistic regression pipeline.
    return Pipeline([
        ("preprocessing", build_preprocessor(
            numeric_features, categorical_features, scale_numeric=True)),
        ("classifier", LogisticRegression(
            max_iter=2000, class_weight="balanced")),
    ])

def build_random_forest_model():
    # Build a regularized Random Forest using the refined feature set.
    return Pipeline([
        ("preprocessing", build_preprocessor(
            refined_numeric_features, refined_categorical_features,
            scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=10,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1)),
    ])

def calculate_classification_metrics(model, X, y, model_name):
    # Return classification metrics at a stated probability threshold.
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= 0.50).astype("int8")
    return {
        "Model": model_name,
        "Rows": len(y),
        "Admission rate (%)": 100 * y.mean(),
        "Average precision": average_precision_score(y, probabilities),
        "ROC-AUC": roc_auc_score(y, probabilities),
        "Balanced accuracy": balanced_accuracy_score(y, predictions),
        "Precision": precision_score(y, predictions, zero_division=0),
        "Recall": recall_score(y, predictions, zero_division=0),
        "F1-score": f1_score(y, predictions, zero_division=0),
    }

# Fit candidate models on the same training observations and score validation.
candidate_models = {
    "Prior-probability reference": DummyClassifier(strategy="prior"),
    "Week 6 features - Logistic Regression": build_logistic_model(
        week6_numeric_features, week6_categorical_features),
    "Refined features - Logistic Regression": build_logistic_model(
        refined_numeric_features, refined_categorical_features),
    "Refined features - Random Forest": build_random_forest_model(),
}

validation_rows = []
for model_name, model in candidate_models.items():
    if model_name == "Prior-probability reference":
        model.fit(np.zeros((len(train_data), 1)),
                  train_data["admission_indicator"])
        validation_X = np.zeros((len(validation_data), 1))
    else:
        if model_name == "Week 6 features - Logistic Regression":
            selected_features = week6_numeric_features + week6_categorical_features
        else:
            selected_features = refined_numeric_features + refined_categorical_features
        model.fit(train_data[selected_features],
                  train_data["admission_indicator"])
        validation_X = validation_data[selected_features]
    validation_rows.append(calculate_classification_metrics(
        model, validation_X, validation_data["admission_indicator"], model_name))

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["Average precision", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(validation_results.round(3))

# Compare training and validation scores to identify possible overfitting.
training_rows = []
for model_name, model in candidate_models.items():
    if model_name == "Prior-probability reference":
        training_X = np.zeros((len(train_data), 1))
    elif model_name == "Week 6 features - Logistic Regression":
        training_X = train_data[
            week6_numeric_features + week6_categorical_features
        ]
    else:
        training_X = train_data[
            refined_numeric_features + refined_categorical_features
        ]
    training_rows.append(calculate_classification_metrics(
        model, training_X, train_data["admission_indicator"], model_name))

training_results = pd.DataFrame(training_rows)
generalization_comparison = training_results[
    ["Model", "Average precision", "ROC-AUC"]
].merge(
    validation_results[["Model", "Average precision", "ROC-AUC"]],
    on="Model",
    suffixes=(" - training", " - validation"),
)
generalization_comparison["Average precision gap"] = (
    generalization_comparison["Average precision - training"]
    - generalization_comparison["Average precision - validation"]
)
generalization_comparison["ROC-AUC gap"] = (
    generalization_comparison["ROC-AUC - training"]
    - generalization_comparison["ROC-AUC - validation"]
)
display(generalization_comparison.round(3))


# Select a candidate from validation data only; the final test remains untouched.
eligible_results = validation_results[
    validation_results["Model"] != "Prior-probability reference"
]
selected_model_name = eligible_results.iloc[0]["Model"]
print("Candidate selected by validation average precision:", selected_model_name)

,Partition,Rows,Start,End,Admission rate (%)
0,Training,3357,2026-01-01,2026-08-28,11.50
1,Validation,822,2026-09-01,2026-10-28,10.95
2,Final test,821,2026-11-01,2026-12-28,13.40


,Model,Rows,Admission rate (%),Average precision,ROC-AUC,Balanced accuracy,Precision,Recall,F1-score
0,Refined features - Logistic Regression,822,10.949,0.135,0.547,0.541,0.127,0.533,0.205
1,Week 6 features - Logistic Regression,822,10.949,0.127,0.512,0.507,0.112,0.467,0.181
2,Refined features - Random Forest,822,10.949,0.117,0.499,0.509,0.137,0.078,0.099
3,Prior-probability reference,822,10.949,0.109,0.500,0.500,0.000,0.000,0.000


,Model,Average precision - training,ROC-AUC - training,Average precision - validation,ROC-AUC - validation,Average precision gap,ROC-AUC gap
0,Prior-probability reference,0.115,0.500,0.109,0.500,0.005,0.000
1,Week 6 features - Logistic Regression,0.153,0.586,0.127,0.512,0.027,0.075
2,Refined features - Logistic Regression,0.165,0.607,0.135,0.547,0.030,0.060
3,Refined features - Random Forest,0.815,0.969,0.117,0.499,0.698,0.470


Candidate selected by validation average precision: Refined features - Logistic Regression


**Interpretation of the development comparison.** On September-October validation, the refined logistic regression has the highest average precision (0.135) and ROC-AUC (0.547). The Week 6 feature specification reaches 0.127 and 0.512; the prior-probability reference has average precision equal to the 0.109 admission prevalence and ROC-AUC 0.500. The Random Forest reaches 0.117 average precision and 0.499 ROC-AUC, so it does not improve the validation ranking.

The training-versus-validation scores provide evidence of overfitting in the Random Forest: its training average precision and ROC-AUC are 0.815 and 0.969, compared with 0.117 and 0.499 on validation. The logistic models have much smaller, though still visible, gaps. This pattern suggests the Random Forest is fitting training-specific structure that does not transfer to later appointments. The validation period is only two months, so this is a warning signal rather than a complete estimate of long-term performance.

The refined logistic regression is selected for the final retest based on validation average precision. The modest validation improvement is not treated as proven until it is compared on the untouched later period.

## Part II - Week 7 Testing and End-to-End Validation

This part tests the selected candidate on November-December 2026, a later period excluded from fitting and model selection. It records expected and actual behavior, analyzes false positives and false negatives, checks performance across basic patient segments, and identifies the evidence still required from the HC-POD before Week 8.

### 2.1 - Testing Readiness and Expected Results

The following checks define the Week 7 evaluation:
- Final test dates must occur after training and validation dates.
- Target and prediction outputs must be complete and binary.
- Predicted probabilities must fall between 0 and 1.
- The refined model must not use admissions_record_count, billing, feedback, tests, or prescriptions as predictors.
- Candidate performance will be compared with the prior-probability reference and the Week 6-style specification.
- Suitability will be based on out-of-time discrimination, precision/recall trade-offs, errors, and segment results; no arbitrary pass threshold will be invented.

A model that does not outperform the prior-probability reference or that produces an unacceptable false-alert burden will be documented as not suitable for operational use.

The comparison of average precision on the final test also includes a paired bootstrap interval. The interval is used to describe uncertainty in the observed difference, not to claim clinical significance. A model-interface check verifies that the fitted preprocessing/model pipeline handles an unseen category and an imputable missing numeric value, and rejects a missing required feature. These are local checks; they do not replace a test of an integrated ML Engineering pipeline.


The next test record states the expected result, actual evidence, pass/fail or pending status, issue, action, and retest result for each major check. Passing code checks do not imply that the model itself is useful; predictive improvement and cross-track integration are assessed separately.


In [5]:
# Refit each candidate using development data, then evaluate on the final period.
development_data = pd.concat([train_data, validation_data], ignore_index=True)
final_test_rows = []
final_fitted_models = {}

for model_name in candidate_models:
    if model_name == "Prior-probability reference":
        final_model = DummyClassifier(strategy="prior")
        final_model.fit(np.zeros((len(development_data), 1)),
                        development_data["admission_indicator"])
        final_test_X = np.zeros((len(test_data), 1))
    else:
        final_model = candidate_models[model_name]
        if model_name == "Week 6 features - Logistic Regression":
            selected_features = week6_numeric_features + week6_categorical_features
        else:
            selected_features = refined_numeric_features + refined_categorical_features
        final_model.fit(development_data[selected_features],
                        development_data["admission_indicator"])
        final_test_X = test_data[selected_features]
    final_fitted_models[model_name] = final_model
    final_test_rows.append(calculate_classification_metrics(
        final_model, final_test_X, test_data["admission_indicator"], model_name))

final_test_results = pd.DataFrame(final_test_rows)
display(final_test_results.round(3))

# Compare the selected candidate with the prior-probability reference.
selected_test_row = final_test_results.loc[
    final_test_results["Model"] == selected_model_name
].iloc[0]
prior_reference_row = final_test_results.loc[
    final_test_results["Model"] == "Prior-probability reference"
].iloc[0]
ranking_improved = (
    selected_test_row["Average precision"]
    > prior_reference_row["Average precision"]
    and selected_test_row["ROC-AUC"] > prior_reference_row["ROC-AUC"]
)
readiness_result = pd.DataFrame([{
    "Selected candidate": selected_model_name,
    "Average precision above prior reference": (
        selected_test_row["Average precision"]
        > prior_reference_row["Average precision"]
    ),
    "ROC-AUC above prior reference": (
        selected_test_row["ROC-AUC"] > prior_reference_row["ROC-AUC"]
    ),
    "Predictive ranking improved on both checks": ranking_improved,
    "Week 7 suitability decision": (
        "Further review required" if ranking_improved
        else "Not suitable for operational use"
    ),
}])
display(readiness_result)


# Prepare predictions from the model selected before looking at the final test.
selected_model = final_fitted_models[selected_model_name]
if selected_model_name == "Week 6 features - Logistic Regression":
    selected_features = week6_numeric_features + week6_categorical_features
else:
    selected_features = refined_numeric_features + refined_categorical_features
X_final_test = test_data[selected_features]
y_final_test = test_data["admission_indicator"]
final_probabilities = selected_model.predict_proba(X_final_test)[:, 1]
final_predictions = (final_probabilities >= 0.50).astype("int8")

if len(final_predictions) != len(test_data):
    raise ValueError("The model did not return one prediction per test row.")
if not np.isfinite(final_probabilities).all():
    raise ValueError("The model returned a missing or infinite probability.")
if ((final_probabilities < 0) | (final_probabilities > 1)).any():
    raise ValueError("Predicted probabilities must be between 0 and 1.")

print(f"Final retest candidate: {selected_model_name}")
print(f"Final test period: {test_data['date'].min().date()} to "
      f"{test_data['date'].max().date()}")
print(f"Final test rows: {len(test_data):,}")

,Model,Rows,Admission rate (%),Average precision,ROC-AUC,Balanced accuracy,Precision,Recall,F1-score
0,Prior-probability reference,821,13.398,0.134,0.500,0.500,0.000,0.000,0.000
1,Week 6 features - Logistic Regression,821,13.398,0.131,0.467,0.469,0.120,0.473,0.192
2,Refined features - Logistic Regression,821,13.398,0.124,0.466,0.471,0.120,0.445,0.190
3,Refined features - Random Forest,821,13.398,0.138,0.493,0.491,0.103,0.055,0.071


,Selected candidate,Average precision above prior reference,ROC-AUC above prior reference,Predictive ranking improved on both checks,Week 7 suitability decision
0,Refined features - Logistic Regression,False,False,False,Not suitable for operational use


Final retest candidate: Refined features - Logistic Regression
Final test period: 2026-11-01 to 2026-12-28
Final test rows: 821


In [6]:

from sklearn.metrics import average_precision_score

# Get probability predictions for each model on the same final test rows.
test_probabilities = {}
for model_name, model in final_fitted_models.items():
    if model_name == "Prior-probability reference":
        model_input = np.zeros((len(test_data), 1))
    elif model_name == "Week 6 features - Logistic Regression":
        model_input = test_data[
            week6_numeric_features + week6_categorical_features
        ]
    else:
        model_input = test_data[
            refined_numeric_features + refined_categorical_features
        ]
    test_probabilities[model_name] = model.predict_proba(model_input)[:, 1]

def bootstrap_average_precision_difference(
    actual, candidate_probability, reference_probability,
    repetitions=1000, random_seed=42
):
    # Resample the same test rows for both models to estimate a paired difference.
    random_generator = np.random.default_rng(random_seed)
    actual = np.asarray(actual)
    candidate_probability = np.asarray(candidate_probability)
    reference_probability = np.asarray(reference_probability)
    differences = []

    for _ in range(repetitions):
        sample_indices = random_generator.integers(
            0, len(actual), size=len(actual)
        )
        sampled_actual = actual[sample_indices]
        if len(np.unique(sampled_actual)) < 2:
            continue
        candidate_score = average_precision_score(
            sampled_actual, candidate_probability[sample_indices]
        )
        reference_score = average_precision_score(
            sampled_actual, reference_probability[sample_indices]
        )
        differences.append(candidate_score - reference_score)

    lower_bound, upper_bound = np.percentile(differences, [2.5, 97.5])
    observed_difference = (
        average_precision_score(actual, candidate_probability)
        - average_precision_score(actual, reference_probability)
    )
    return observed_difference, lower_bound, upper_bound

# Check whether the observed average-precision differences are stable enough
# to distinguish from zero on this limited test sample.
comparison_names = [
    "Prior-probability reference",
    "Week 6 features - Logistic Regression",
]
bootstrap_rows = []
for reference_name in comparison_names:
    difference, lower, upper = bootstrap_average_precision_difference(
        y_final_test,
        test_probabilities[selected_model_name],
        test_probabilities[reference_name],
    )
    bootstrap_rows.append({
        "Comparison": f"Selected candidate minus {reference_name}",
        "Average precision difference": difference,
        "95% bootstrap lower bound": lower,
        "95% bootstrap upper bound": upper,
        "Interval excludes zero": lower > 0 or upper < 0,
    })
bootstrap_comparison = pd.DataFrame(bootstrap_rows)
display(bootstrap_comparison.round(3))

# Verify the selected model's behavior on missing and unseen category values.
model_input = X_final_test.iloc[[0]].copy()
unknown_category_input = model_input.copy()
for column in refined_categorical_features:
    unknown_category_input[column] = "__unseen_category__"
unknown_category_probability = selected_model.predict_proba(
    unknown_category_input
)[0, 1]

missing_value_input = model_input.copy()
missing_value_input[refined_numeric_features[0]] = np.nan
missing_value_probability = selected_model.predict_proba(
    missing_value_input
)[0, 1]

missing_column_input = model_input.drop(
    columns=[selected_features[0]]
)
try:
    selected_model.predict_proba(missing_column_input)
    missing_required_column_rejected = False
except (KeyError, ValueError):
    missing_required_column_rejected = True

pipeline_contract_results = pd.DataFrame([
    {"Input scenario": "Unseen category", "Expected behavior": "Process safely",
     "Actual result": f"Probability {unknown_category_probability:.3f}",
     "Status": "Pass" if 0 <= unknown_category_probability <= 1 else "Fail"},
    {"Input scenario": "Missing numeric value", "Expected behavior": "Impute and predict",
     "Actual result": f"Probability {missing_value_probability:.3f}",
     "Status": "Pass" if 0 <= missing_value_probability <= 1 else "Fail"},
    {"Input scenario": "Missing required column", "Expected behavior": "Reject input clearly",
     "Actual result": "Input rejected" if missing_required_column_rejected else "Input accepted",
     "Status": "Pass" if missing_required_column_rejected else "Fail"},
])
display(pipeline_contract_results)


,Comparison,Average precision difference,95% bootstrap lower bound,95% bootstrap upper bound,Interval excludes zero
0,Selected candidate minus Prior-probability ref...,-0.010,-0.023,0.017,False
1,Selected candidate minus Week 6 features - Log...,-0.007,-0.028,0.008,False


,Input scenario,Expected behavior,Actual result,Status
0,Unseen category,Process safely,Probability 0.471,Pass
1,Missing numeric value,Impute and predict,Probability 0.426,Pass
2,Missing required column,Reject input clearly,Input rejected,Pass


In [7]:

# Create the required expected-versus-actual testing and retest record.
selected_test_row = final_test_results.loc[
    final_test_results["Model"] == selected_model_name
].iloc[0]
prior_test_row = final_test_results.loc[
    final_test_results["Model"] == "Prior-probability reference"
].iloc[0]
selected_bootstrap = bootstrap_comparison.iloc[0]
contract_status = dict(zip(
    pipeline_contract_results["Input scenario"],
    pipeline_contract_results["Status"],
))

test_record = pd.DataFrame([
    {
        "Test / scenario": "Target and appointment grain",
        "Expected result": "Unique appointments and a complete binary target",
        "Actual result": "5,000 unique appointments; 586 admissions; no invalid target values",
        "Status": "Pass",
        "Issue identified": "Admission prevalence is 11.7%",
        "Action taken": "Use imbalance-aware metrics and a prior-probability reference",
        "Retest result": "Checks passed on the executed modeling dataset",
    },
    {
        "Test / scenario": "Direct target leakage",
        "Expected result": "No target copy appears among model inputs",
        "Actual result": "Admissions record count equals the target on all rows and is excluded",
        "Status": "Pass",
        "Issue identified": "Week 6 master contains a direct target duplicate",
        "Action taken": "Prohibit admissions record count from the predictor lists",
        "Retest result": "Candidate pipelines fit and predicted without this column",
    },
    {
        "Test / scenario": "Chronological partition",
        "Expected result": "Train, validation, and final test dates do not overlap",
        "Actual result": "Jan-Aug train; Sep-Oct validation; Nov-Dec final test",
        "Status": "Pass",
        "Issue identified": "Only one year of observations is available",
        "Action taken": "Keep the final two months untouched until retesting",
        "Retest result": "Final test contains 821 later appointments",
    },
    {
        "Test / scenario": "Unseen category handling",
        "Expected result": "Pipeline returns a bounded probability",
        "Actual result": contract_status["Unseen category"],
        "Status": contract_status["Unseen category"],
        "Issue identified": "New categories may occur at inference time",
        "Action taken": "Use one-hot encoding with unknown categories ignored",
        "Retest result": "Local prediction returned a probability in [0, 1]",
    },
    {
        "Test / scenario": "Missing optional numeric value",
        "Expected result": "Pipeline imputes the value and returns a bounded probability",
        "Actual result": contract_status["Missing numeric value"],
        "Status": contract_status["Missing numeric value"],
        "Issue identified": "An optional numeric input may be absent",
        "Action taken": "Use median imputation learned from training data",
        "Retest result": "Local prediction succeeded and returned a bounded probability",
    },
    {
        "Test / scenario": "Missing required input column",
        "Expected result": "Pipeline rejects an invalid input rather than silently predicting",
        "Actual result": contract_status["Missing required column"],
        "Status": contract_status["Missing required column"],
        "Issue identified": "An incomplete feature schema cannot support a valid prediction",
        "Action taken": "Keep required model columns explicit in the contract",
        "Retest result": "Local pipeline rejected the incomplete input",
    },
    {
        "Test / scenario": "Predictive improvement on final test",
        "Expected result": "Refined candidate improves ranking over prior and Week 6 comparator",
        "Actual result": (
            f"Average precision {selected_test_row['Average precision']:.3f}; "
            f"ROC-AUC {selected_test_row['ROC-AUC']:.3f}; "
            f"prior average precision {prior_test_row['Average precision']:.3f}"
        ),
        "Status": "Fail",
        "Issue identified": "The selected candidate does not beat the prior reference",
        "Action taken": "Do not approve for operational use; investigate target and features",
        "Retest result": "Refined model remains below the reference on this test period",
    },
    {
        "Test / scenario": "Meaningful average-precision improvement",
        "Expected result": "Paired 95% bootstrap interval for the gain is above zero",
        "Actual result": (
            f"Difference {selected_bootstrap['Average precision difference']:.3f}; "
            f"95% interval [{selected_bootstrap['95% bootstrap lower bound']:.3f}, "
            f"{selected_bootstrap['95% bootstrap upper bound']:.3f}]"
        ),
        "Status": "Fail",
        "Issue identified": "The interval includes zero; improvement is not established",
        "Action taken": "Report uncertainty and avoid claiming meaningful uplift",
        "Retest result": "No statistically distinguishable gain over the prior reference",
    },
    {
        "Test / scenario": "Attendance/no-show use case",
        "Expected result": "A validated attendance outcome is available",
        "Actual result": "No no-show label; the status field is Active for every row",
        "Status": "Fail",
        "Issue identified": "Current target is admission, not missed appointment",
        "Action taken": "Keep admission scope explicit and request validated attendance data",
        "Retest result": "Blocked until a suitable no-show target is supplied",
    },
    {
        "Test / scenario": "HC-POD cross-track integration",
        "Expected result": "Partner test, finding, action, retest, and evidence are recorded",
        "Actual result": "No partner artifact or completed collaboration supplied",
        "Status": "Pending",
        "Issue identified": "Integrated ML Engineering pipeline behavior is unverified",
        "Action taken": "Share the model contract and arrange a real partner test",
        "Retest result": "Pending receipt of partner output and evidence",
    },
])
display(test_record)


,Test / scenario,Expected result,Actual result,Status,Issue identified,Action taken,Retest result
0,Target and appointment grain,Unique appointments and a complete binary target,"5,000 unique appointments; 586 admissions; no ...",Pass,Admission prevalence is 11.7%,Use imbalance-aware metrics and a prior-probab...,Checks passed on the executed modeling dataset
1,Direct target leakage,No target copy appears among model inputs,Admissions record count equals the target on a...,Pass,Week 6 master contains a direct target duplicate,Prohibit admissions record count from the pred...,Candidate pipelines fit and predicted without ...
2,Chronological partition,"Train, validation, and final test dates do not...",Jan-Aug train; Sep-Oct validation; Nov-Dec fin...,Pass,Only one year of observations is available,Keep the final two months untouched until rete...,Final test contains 821 later appointments
3,Unseen category handling,Pipeline returns a bounded probability,Pass,Pass,New categories may occur at inference time,Use one-hot encoding with unknown categories i...,"Local prediction returned a probability in [0, 1]"
4,Missing optional numeric value,Pipeline imputes the value and returns a bound...,Pass,Pass,An optional numeric input may be absent,Use median imputation learned from training data,Local prediction succeeded and returned a boun...
5,Missing required input column,Pipeline rejects an invalid input rather than ...,Pass,Pass,An incomplete feature schema cannot support a ...,Keep required model columns explicit in the co...,Local pipeline rejected the incomplete input
6,Predictive improvement on final test,Refined candidate improves ranking over prior ...,Average precision 0.124; ROC-AUC 0.466; prior ...,Fail,The selected candidate does not beat the prior...,Do not approve for operational use; investigat...,Refined model remains below the reference on t...
7,Meaningful average-precision improvement,Paired 95% bootstrap interval for the gain is ...,"Difference -0.010; 95% interval [-0.023, 0.017]",Fail,The interval includes zero; improvement is not...,Report uncertainty and avoid claiming meaningf...,No statistically distinguishable gain over the...
8,Attendance/no-show use case,A validated attendance outcome is available,No no-show label; the status field is Active f...,Fail,"Current target is admission, not missed appoin...",Keep admission scope explicit and request vali...,Blocked until a suitable no-show target is sup...
9,HC-POD cross-track integration,"Partner test, finding, action, retest, and evi...",No partner artifact or completed collaboration...,Pending,Integrated ML Engineering pipeline behavior is...,Share the model contract and arrange a real pa...,Pending receipt of partner output and evidence


**Interpretation of the Week 7 retest.** The final test contains 821 appointments, including 110 admissions (13.4%). The selected refined logistic regression reaches average precision 0.124, below the test prevalence of 0.134, and ROC-AUC 0.466. The Week 6-style logistic comparator reaches 0.131 average precision and 0.467 ROC-AUC; the prior reference has 0.134 and 0.500. Thus the selected model does not outperform either comparator on this later period.

The Random Forest has average precision 0.138, only slightly above prevalence, but ROC-AUC 0.493 and recall 5.5% at the fixed 0.50 threshold. Its very high training scores and near-random validation/test results show severe overfitting. The paired bootstrap interval for the selected model's average-precision difference versus the prior reference is -0.023 to 0.017; versus the Week 6-style comparator it is -0.028 to 0.008. Both intervals include zero, so the observed differences do not establish meaningful improvement.

### 2.2 - False-Positive, False-Negative, and Segment Analysis

A useful model review should show which errors occur, not only aggregate scores. The next cell summarizes the confusion matrix and reports recall, precision, and balanced accuracy by age group and gender. Segment results are descriptive; small groups and the short final test period limit conclusions about fairness or stable subgroup performance.

In [8]:
from sklearn.metrics import confusion_matrix

# Count correct and incorrect predictions on the final test period.
confusion = confusion_matrix(y_final_test, final_predictions, labels=[0, 1])
confusion_table = pd.DataFrame(
    confusion,
    index=["Actual no admission", "Actual admission"],
    columns=["Predicted no admission", "Predicted admission"],
)
display(confusion_table)

true_negative, false_positive, false_negative, true_positive = confusion.ravel()
error_summary = pd.DataFrame([
    {"Error or outcome": "True negatives", "Rows": true_negative},
    {"Error or outcome": "False positives", "Rows": false_positive},
    {"Error or outcome": "False negatives", "Rows": false_negative},
    {"Error or outcome": "True positives", "Rows": true_positive},
])
display(error_summary)

# Check whether errors concentrate in particular departments or diagnoses.
error_frame = test_data[["departmentname", "diagnoses"]].copy()
error_frame["actual"] = y_final_test.to_numpy()
error_frame["predicted"] = final_predictions
error_frame["false_positive"] = (
    error_frame["actual"].eq(0) & error_frame["predicted"].eq(1)
).astype("int8")
error_frame["false_negative"] = (
    error_frame["actual"].eq(1) & error_frame["predicted"].eq(0)
).astype("int8")

for grouping_column in ["departmentname", "diagnoses"]:
    grouped_errors = error_frame.groupby(grouping_column).agg(
        Appointments=("actual", "size"),
        Actual_admissions=("actual", "sum"),
        False_positives=("false_positive", "sum"),
        False_negatives=("false_negative", "sum"),
    )
    grouped_errors["False_positive_rate"] = (
        grouped_errors["False_positives"]
        / (grouped_errors["Appointments"] - grouped_errors["Actual_admissions"])
    )
    grouped_errors["False_negative_rate"] = (
        grouped_errors["False_negatives"]
        / grouped_errors["Actual_admissions"].replace(0, np.nan)
    )
    print(f"Error patterns by {grouping_column}:")
    display(grouped_errors.sort_values(
        "False_positive_rate", ascending=False
    ).round(3))


# Add age groups for a limited descriptive segment check.
segment_frame = test_data[["age", "gender"]].copy()
segment_frame["actual"] = y_final_test.to_numpy()
segment_frame["predicted"] = final_predictions
segment_frame["age_group"] = pd.cut(
    segment_frame["age"],
    bins=[-1, 17, 34, 49, 64, np.inf],
    labels=["0-17", "18-34", "35-49", "50-64", "65+"],
)

def summarize_segment(group, segment_name, segment_value):
    # Calculate interpretable metrics for one sufficiently large segment.
    actual = group["actual"]
    predicted = group["predicted"]
    return {
        "Segment": segment_name,
        "Group": str(segment_value),
        "Rows": len(group),
        "Admissions": int(actual.sum()),
        "Admission rate (%)": 100 * actual.mean(),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Balanced accuracy": (
            balanced_accuracy_score(actual, predicted)
            if actual.nunique() == 2 else np.nan
        ),
    }

segment_rows = []
for segment_name in ["age_group", "gender"]:
    for segment_value, group in segment_frame.groupby(
        segment_name, observed=True
    ):
        if len(group) >= 30:
            segment_rows.append(
                summarize_segment(group, segment_name, segment_value)
            )
segment_results = pd.DataFrame(segment_rows)
display(segment_results.round(3))

,Predicted no admission,Predicted admission
Actual no admission,353,358
Actual admission,61,49


,Error or outcome,Rows
0,True negatives,353
1,False positives,358
2,False negatives,61
3,True positives,49


Error patterns by departmentname:


,Appointments,Actual_admissions,False_positives,False_negatives,False_positive_rate,False_negative_rate
departmentname,,,,,,
Pharmacy,72,14,44,2,0.759,0.143
Surgery,91,12,59,6,0.747,0.500
Cardiology,95,14,54,7,0.667,0.500
Emergency,102,9,57,5,0.613,0.556
Laboratory,73,11,34,6,0.548,0.545
Obstetrics & Gynecology,80,11,35,6,0.507,0.545
Radiology,88,8,37,1,0.462,0.125
Icu,77,10,18,8,0.269,0.800
Pediatrics,73,11,13,11,0.210,1.000


Error patterns by diagnoses:


,Appointments,Actual_admissions,False_positives,False_negatives,False_positive_rate,False_negative_rate
diagnoses,,,,,,
Gastritis,165,18,99,7,0.673,0.389
Upper Respiratory Infection,178,22,99,8,0.635,0.364
Hypertension,188,27,84,17,0.522,0.630
Diabetes,156,23,46,17,0.346,0.739
Malaria,134,20,30,12,0.263,0.600


,Segment,Group,Rows,Admissions,Admission rate (%),Recall,Precision,Balanced accuracy
0,age_group,0-17,173,23,13.295,0.435,0.145,0.521
1,age_group,18-34,145,25,17.241,0.200,0.096,0.404
2,age_group,35-49,159,14,8.805,0.429,0.075,0.459
3,age_group,50-64,126,24,19.048,0.542,0.183,0.487
4,age_group,65+,218,24,11.009,0.625,0.111,0.503
5,gender,Female,371,45,12.129,0.467,0.151,0.552
6,gender,Male,450,65,14.444,0.431,0.104,0.404


**Interpretation of errors and segments.** For the selected refined logistic regression at the 0.50 threshold, the test has 353 true negatives, 358 false positives, 61 false negatives, and 49 true positives. It raises 407 alerts, of which only 49 are correct: precision is 12.0%, recall is 44.5%. This creates a substantial administrative follow-up burden while missing more than half of admissions.

Error rates also vary across groups. False-positive rates are highest for Pharmacy (75.9%) and Surgery (74.7%); false-negative rates are high for Pediatrics (100%), Internal Medicine (90%), and ICU (80%). By diagnosis, false-positive rates are highest for Gastritis (67.3%) and Upper Respiratory Infection (63.5%). These rates are based on a two-month holdout with small admission counts per group. They identify review priorities but do not support stable subgroup or fairness claims.

The age and gender table shows recall from 20.0% for ages 18-34 to 62.5% for ages 65+, and 43.1% for males versus 46.7% for females. These are further warning signals to investigate with more data and stakeholder review.

### 2.3 - HC-POD Cross-Track Validation Record and ML Engineering Handoff

The Week 7 brief requires a meaningful cross-track activity with evidence of a test, finding, action, retest, and the exchanged information. Data Science has now received a response from Agu Precious in the Data Analytics track. She reports that she validated the billing trend, outstanding balances, and complaint rate; she also reports that facility and department rating differences were not statistically significant and should not be treated as features. She confirms that the dataset has no attendance/no-show field. These are partner-reported findings; the underlying analyses have not been independently rerun in this notebook. The exchange is genuine collaboration evidence, but a joint model-feature review and retest are still pending. The local model-interface checks are not an integrated pipeline test.

#### Model Contract Prepared for ML Engineering

| Contract element | Current specification |
|---|---|
| Intended prediction point | After initial diagnosis/severity are documented and before admission outcome; timestamp assumption requires confirmation |
| Required numeric inputs | age, previous_appointment_count, previous_admission_count, previous_admission_rate, disease_severity_score |
| Required categorical inputs | gender, visittype, departmentname, chroniccondition, diagnoses, appointment_month, appointment_day_of_week |
| Preprocessing behavior | Median imputation and scaling for numeric inputs; most-frequent imputation and one-hot encoding for categories; unseen categories are ignored |
| Output | Admission probability from 0 to 1 and class at the documented 0.50 threshold |
| Safety and use restriction | Analytical prototype only; not for clinical decisions or no-show prediction |

#### Cross-Track Evidence Record

| Required record | Actual status |
|---|---|
| 1. Track collaborated with | Data Analytics; Agu Precious (response received). ML Engineering integration remains a separate pending activity |
| 2. Project dependency | Model feature schema and prediction output must match the integrated pipeline |
| 3. Component/output tested | Data Analytics reviewed the supplied data for billing, balance, complaint, and rating findings; a joint review of this admission model has not yet been run |
| 4. Information/output received | Partner reports validation of billing trend, outstanding balances, and complaint rate; facility/department rating differences were not significant; no attendance/no-show field exists. Findings are reported by the partner and await artifact-level review |
| 5. Information/output provided | The partner requested the target and feature list. This notebook documents them below; sending the reply is still pending and must not be reported as completed |
| 6. Testing activity completed | Data Analytics performed/reported validation on the listed operational metrics and rating differences. A feature-support review using the target and schema below is pending; local pipeline edge-case checks are complete |
| 7. Issue or finding identified | No no-show target; rating differences do not support rating features; target/feature support and event timing still need joint confirmation |
| 8. Refinement/action taken | No rating variables are used. Admission remains the explicit target; no-show prediction is out of scope. Leaking/constant predictors were removed and tests/prescriptions excluded pending timing evidence |
| 9. Retest result | Local interface checks passed. Cross-track feature review and any resulting model retest remain pending |
| 10. What changed as a result | The collaboration record now includes Data Analytics findings and a request to exchange the target/schema. No model change is attributed to the partner before the joint support review |
| 11. Evidence | Data Analytics message received in this project conversation; retain the message or an approved screenshot in the project evidence folder. Local test outputs are in the notebook |
| 12. Contribution to HealthConnect | Connects Data Analytics findings to feature selection, prevents unsupported rating features, and keeps the missing attendance target visible; joint retest and end-to-end validation remain incomplete |

#### Target and Feature List Requested by Data Analytics

| Item | Current modeling definition |
|---|---|
| Target | `admission_indicator` (0 = no admission recorded for the appointment; 1 = admission recorded). It is derived from the source admission records and still requires data-owner confirmation. It is not an attendance/no-show target. |
| Numeric predictors | `age`, `previous_appointment_count`, `previous_admission_count`, `previous_admission_rate`, `disease_severity_score` |
| Categorical predictors | `gender`, `visittype`, `departmentname`, `chroniccondition`, `diagnoses`, `appointment_month`, `appointment_day_of_week` |
| Excluded fields | `admissions_record_count` (duplicates the target), `diagnoses_record_count` (constant), same-visit laboratory and prescription counts (event timing unknown), facility/department rating measures (partner reports non-significant differences; not model features), billing/revenue fields (not part of this admission model) |

The Data Analytics response supports keeping rating measures out of the feature set; `departmentname` remains a descriptive care-context variable and is not a rating. Billing trends, outstanding balances, and complaint rate are useful findings for the broader project, but they answer different questions and should not be added to an admission model without a clear causal/time-of-prediction rationale and a separate validation.

**Next collaboration step:** send the target and feature list above to Agu, ask her to confirm which fields and event timestamps are supported by the source data, and request the specific validation artifact or summary for her reported findings. Then record the actual response, change the feature set only when justified, rerun the same temporal validation, and document the retest. Separately, share the model contract with ML Engineering and retain its integrated test output.

### 2.4 - Model Suitability, Business Relevance, Risks, and Week 8 Readiness

**Week 7 decision: the admission model is not suitable for operational use.** On the later test period, the selected refined logistic regression has lower average precision than both the admission prevalence and the Week 6-style comparator, with ROC-AUC 0.466. The Random Forest is also weak (ROC-AUC 0.493) and detects few admissions at the stated threshold. The refinement did not establish a meaningful, generalizable improvement.

**Business relevance.** The tested output estimates admission, not appointment attendance. Its large false-positive count could create unnecessary administrative follow-up, while missed admissions show that many positive outcomes are still not identified. Since there is no no-show outcome, this model cannot currently support the stated missed-appointment reduction objective. Its useful contribution this week is identifying the target, timing, and data gaps that must be resolved before deployment work.

**Limitations and risks.** The admission target is derived from admissions_record_count and needs source-owner confirmation. Diagnosis and severity timestamps are unavailable. The temporal holdout covers only November-December 2026, and performance varies across small segments. Department-to-facility mapping was inconsistent in Week 6, so facility-level predictors are excluded. Local input checks do not validate an integrated ML Engineering service. Model outputs are not clinical validation and must not determine care.

**Before Week 8:** obtain a validated attendance/no-show label if attendance prediction remains required; confirm the event timing and target definition with the data owner; complete the ML Engineering or Data Analytics cross-track test with real evidence; define the cost/capacity trade-off for false alerts with stakeholders; and retain this later-period result as the current model-readiness decision.

## Week 7 Project Summary

| Required summary item | Week 7 record |
|---|---|
| 1. What was planned to be tested | The Week 6 admission classifier, its features, errors, later-period performance, and model input/output behavior |
| 2. What was actually tested | Week 6-style logistic regression, refined logistic regression, Random Forest, prior-probability reference, later-period holdout, and local input edge cases |
| 3. Most important testing results | Refined logistic: average precision 0.124 and ROC-AUC 0.466 on the final set; admission prevalence 0.134 |
| 4. Issues or weaknesses identified | Weak out-of-time ranking, many false alerts, missing no-show target, uncertain event timing, and no integrated pipeline artifact |
| 5. Improvements/refinements made | Removed the constant diagnosis count and target-leaking admission count; excluded unverified downstream tests/prescriptions; engineered prior history and used chronological partitions |
| 6. Retesting results | The refined model did not beat the prior reference or Week 6-style model on the final period; Random Forest was also weak |
| 7. Track(s) collaborated with | Data Analytics: Agu Precious replied with data findings; ML Engineering integration remains pending |
| 8. What was tested collaboratively | Data Analytics reports validation of billing trend, outstanding balances, complaint rate, and rating differences; joint review of this model’s target/features and a retest are pending |
| 9. What changed as a result | The findings and requested target/feature exchange are now recorded; ratings remain excluded and no partner-driven model change is claimed before joint review |
| 10. Key findings or validation outcomes | No candidate shows convincing predictive signal for operational admission support; no no-show prediction is possible with the current target |
| 11. Major challenges encountered | Only one year of data, 11.7% overall admission prevalence, absent event timestamps, and no Week 5 HealthConnect baseline |
| 12. Important decisions made | Keep the Week 6 score as historical reference; use a common temporal comparison; reserve Nov-Dec for final evaluation; do not deploy |
| 13. Remaining limitations | Two-month test horizon, target provenance, feature timing assumptions, subgroup uncertainty, and no clinical validation |
| 14. Remaining issues or dependencies | No-show label, data-owner confirmation, target/feature support review with Data Analytics, its evidence artifact, ML Engineering integration test, and business threshold/capacity definition |
| 15. Contribution to HealthConnect | Audited the candidate modeling path and documented changes, failures, integration contract, and use restrictions |
| 16. What must be completed before Week 8 | Send the target/schema to Agu and document her validation response; resolve target/timing questions, complete cross-track retest and ML Engineering integration where available, and determine whether attendance data are needed |

This summary records a real Data Analytics exchange and distinguishes it from the still-pending joint feature retest and ML Engineering integration test.

### Week 7 Data Science Task Traceability

| Assignment task | Notebook evidence | Status |
|---|---|---|
| 1. Review Week 6 candidate | Section 1.1 and historical metrics | Complete |
| 2. Review Week 6 testing requirements | Section 1.1 identifies existing holdout evidence and gaps | Complete |
| 3. Define testing approach | Sections 1.4 and 2.1 | Complete |
| 4. Test with appropriate data | Chronological train/validation/final test design | Complete |
| 5. Evaluate classification metrics | Average precision, ROC-AUC, balanced accuracy, precision, recall, F1 | Complete |
| 6. Analyze model errors | Section 2.2 confusion matrix and grouped error rates | Complete |
| 7. Investigate false positives/negatives | Counts, rates, and department/diagnosis patterns | Complete |
| 8. Identify error patterns | Age, gender, department, and diagnosis summaries | Complete |
| 9. Assess relevant segments | Age and gender performance table | Complete, exploratory only |
| 10. Review feature relevance | Section 1.3 constant, leaking, timing, and history review | Complete |
| 11. Assess overfitting/model weakness | Training-validation metric gaps in Section 1.4 | Complete, limited validation window |
| 12. Compare against Week 5 baseline | No HealthConnect Week 5 model exists; compare actual Week 6 reference transparently | Context mismatch documented |
| 13. Assess meaningful improvement | Paired bootstrap interval for average-precision differences | Complete; interpret uncertainty |
| 14. Identify remaining weaknesses | Section 2.4 | Complete |
| 15. Incorporate Data Analytics findings | Agu Precious reports validation of billing/balance/complaint trends and non-significant rating differences; joint target/feature support review and saved evidence artifact pending | Partial; partner findings received, model retest pending |
| 16. Communicate requirements/results to ML Engineering | Model contract prepared in Section 2.3 | Prepared; transmission and integration test not evidenced |
| 17. Make justified refinements | Section 1.3 | Complete |
| 18. Re-test refined model | Section 2.1 on November-December holdout | Complete |
| 19. Document before/after results | Week 6 reference, controlled validation comparison, and final retest | Complete; split differences disclosed |
| 20. Assess suitability | Section 2.4 | Complete; not suitable for operational use |
| 21. Document limitations and risks | Section 2.4 | Complete |
| 22. Define Week 8 work | Section 2.4 | Complete |